# 14 - Word2Vec with gaianet/london Dataset

Goal: Train Word2Vec on gaianet/london corpus from HuggingFace

Run with: conda activate tfenv

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import plotly.express as px
import pandas as pd
import time

print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1779040983.955838     675 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779040984.102365     675 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779040987.263746     675 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0


In [2]:
# Load dataset using HuggingFace datasets API
print("Loading gaianet/london dataset...")
from datasets import load_dataset

ds = load_dataset("gaianet/london", split="train")

# Extract text
print(f"Dataset size: {len(ds)}")
texts = [row['text'] if 'text' in row else row.get('content', '') for row in ds][:10000]
full_text = ' '.join(texts[:5000])
print(f"Total chars: {len(full_text)}")

Loading gaianet/london dataset...


/home/eanorambuena/miniconda/envs/tfenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 661/661 [00:00<00:00, 16721.36 examples/s]

Dataset size: 661
Total chars: 186582


In [3]:
print(f"Total words: {len(full_text.split())}")

Total words: 29926


In [4]:
# Build vocabulary
words = full_text.lower().split()
words = [w.strip('.,;:!?()[]"\'-0123456789') for w in words]
words = [w for w in words if len(w) > 2]

from collections import Counter
word_counts = Counter(words)
vocab = [w for w, c in word_counts.items() if c >= 5]

text_vocab = {word: idx for idx, word in enumerate(vocab)}
text_vocab_size = len(text_vocab)
text_embed_dim = 64

print(f"Vocabulary: {text_vocab_size} words")

Vocabulary: 807 words


In [5]:
# Create training pairs (Skip-gram)
def create_pairs(words, window=2):
    pairs = []
    for i, word in enumerate(words):
        if word not in text_vocab:
            continue
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j and words[j] in text_vocab:
                pairs.append((text_vocab[word], text_vocab[words[j]]))
    return pairs

pairs = create_pairs(words)
train_words = np.array([p[0] for p in pairs], dtype=np.int32)
train_context = np.array([p[1] for p in pairs], dtype=np.int32)
print(f"Training pairs: {len(pairs)}")

Training pairs: 54998


In [6]:
# Word2Vec from scratch using Keras
embedding_layer = layers.Embedding(
    input_dim=text_vocab_size,
    output_dim=text_embed_dim,
    name='embedding'
)

model = keras.Sequential([
    embedding_layer,
    layers.Flatten(),
    layers.Dense(text_vocab_size, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.02),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Word2Vec model ready!")
model.summary()

Word2Vec model ready!


E0000 00:00:1779041000.552624     675 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1779041000.581727     675 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Efficient training with tf.data
train_ds = tf.data.Dataset.from_tensor_slices((train_words, train_context))
train_ds = train_ds.shuffle(buffer_size=len(train_words)).batch(512).prefetch(tf.data.AUTOTUNE)

print("Training Word2Vec...")
start = time.time()

history = model.fit(
    train_ds,
    epochs=10,
    verbose=1
)

print(f"Training time: {time.time() - start:.2f}s")

Training Word2Vec...
Epoch 1/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1293 - loss: 5.5075
Epoch 2/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1333 - loss: 4.6620
Epoch 3/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1248 - loss: 4.3421
Epoch 4/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1229 - loss: 4.2377
Epoch 5/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1211 - loss: 4.1855
Epoch 6/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1223 - loss: 4.1559
Epoch 7/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1239 - loss: 4.1347
Epoch 8/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1218 - loss: 4.1202
Epoch 9/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1219 - loss: 4.1077
Epoch 10/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1233 - loss: 4.0985
Training time: 12.62s


In [8]:
# Get embeddings
embeddings = embedding_layer.get_weights()[0]
print(f"Embeddings shape: {embeddings.shape}")

# Find similar words
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

def find_similar(word, top_n=5):
    if word not in text_vocab:
        return []
    word_idx = text_vocab[word]
    word_emb = embeddings[word_idx]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("=== Similar Words ===")
test_words = ['tecnología', 'datos', 'desarrollo', 'empresa', 'aprendizaje', 'vida']
for word in test_words:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' -> {[(w, f'{s:.3f}') for w, s in similar]}")

Embeddings shape: (807, 64)
=== Similar Words ===


In [9]:
# Visualize in 3D with PCA
from sklearn.decomposition import PCA

top_words = [w for w, c in word_counts.most_common(200) if w in text_vocab]
top_indices = [text_vocab[w] for w in top_words]
top_embeddings = embeddings[top_indices]

pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(top_embeddings)

df = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df['word'] = top_words

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                 title='Word Embeddings (PCA 3D) - Top 200 words')
fig.update_traces(marker=dict(size=6), textposition='top center')
fig.show()